In [13]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
## dados

especialidades = [i for i in range(10)]
idades = [i for i in range(18,71)]
urgencias = [i for i in range(1,11)]
duracao = [i for i in range(15,65,5)]
experiencia = [i for i in range(5,10)]
carga_max = [i for i in range(6,15,2)]

In [3]:
print(especialidades)
print(idades)
print(urgencias)
print(duracao)
print(experiencia)
print(carga_max)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
[5, 6, 7, 8, 9]
[6, 8, 10, 12, 14]


In [29]:
medicos = {
    f'medico_{i}':{
        'especialidades':np.unique(np.random.choice(especialidades,np.random.choice([1,2,3]))), 
        'experiencia':(exp := np.random.choice(experiencia)),
        'idade':exp+np.random.choice(idades),  
        'carga_max':np.random.choice(carga_max)
}
for i in range(10)
}

pacientes = {
    f'paciente_{i}':{
        'idade':np.random.choice(idades),
        'urgencia':np.random.choice(urgencias),
        'duracao':np.random.choice(duracao),
        'necessidade':np.unique(np.random.choice(especialidades,np.random.choice([1,2,3])))
}
for i in range(10)
}



In [30]:
pacientes

{'paciente_0': {'idade': np.int64(53),
  'urgencia': np.int64(3),
  'duracao': np.int64(55),
  'necessidade': array([1, 7, 9])},
 'paciente_1': {'idade': np.int64(68),
  'urgencia': np.int64(1),
  'duracao': np.int64(55),
  'necessidade': array([0, 5, 7])},
 'paciente_2': {'idade': np.int64(47),
  'urgencia': np.int64(4),
  'duracao': np.int64(15),
  'necessidade': array([7, 9])},
 'paciente_3': {'idade': np.int64(24),
  'urgencia': np.int64(7),
  'duracao': np.int64(30),
  'necessidade': array([2, 8])},
 'paciente_4': {'idade': np.int64(44),
  'urgencia': np.int64(4),
  'duracao': np.int64(25),
  'necessidade': array([1, 7])},
 'paciente_5': {'idade': np.int64(34),
  'urgencia': np.int64(9),
  'duracao': np.int64(45),
  'necessidade': array([5])},
 'paciente_6': {'idade': np.int64(61),
  'urgencia': np.int64(9),
  'duracao': np.int64(25),
  'necessidade': array([3])},
 'paciente_7': {'idade': np.int64(54),
  'urgencia': np.int64(6),
  'duracao': np.int64(30),
  'necessidade': array([4

In [57]:
#arcos possiveis
relacao={}
for i in range(len(medicos)):
    # print(f"MEDICO EPECIALIDADES: {medicos[f'medico_{i}']['especialidades']}")
    for j in range(len(pacientes)):
        # print(f"Paciente {j} Necessidade: {pacientes[f'paciente_{j}']['necessidade']}")
        for n in pacientes[f'paciente_{j}']['necessidade']:
            # print(n)
            # print(medicos[f'medico_{i}']['especialidades'])
            if n in medicos[f'medico_{i}']['especialidades']:
                relacao[(1,2)]= n
                # print(medicos[f'medico_{i}']['especialidades'])
        

In [18]:
model = pyo.ConcreteModel()

model.med = pyo.Set(initialize=list(medicos.keys()))
model.pac = pyo.Set(initialize=list(pacientes.keys()))

model.x = pyo.Var(model.med,model.pac, domain=pyo.Binary)


In [19]:
model.pprint()

2 Set Declarations
    med : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :   10 : {'medico_0', 'medico_1', 'medico_2', 'medico_3', 'medico_4', 'medico_5', 'medico_6', 'medico_7', 'medico_8', 'medico_9'}
    pac : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :   10 : {'paciente_0', 'paciente_1', 'paciente_2', 'paciente_3', 'paciente_4', 'paciente_5', 'paciente_6', 'paciente_7', 'paciente_8', 'paciente_9'}

1 Var Declarations
    x : Size=100, Index=med*pac
        Key                        : Lower : Value : Upper : Fixed : Stale : Domain
        ('medico_0', 'paciente_0') :     0 :  None :     1 : False :  True : Binary
        ('medico_0', 'paciente_1') :     0 :  None :     1 : False :  True : Binary
        ('medico_0', 'paciente_2') :     0 :  None :     1 : False :  True : Binary
        ('medico_0', 'paciente_3') :     0 :  None :     1 : Fa

In [ ]:
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=False)
